# Unidade IV — Mineração de Padrões

## Itemsets frequentes e regras de associação

**Carga estimada:** 3 horas  
**Pré-requisitos:** conjuntos, probabilidade condicional e pandas.

> **Pergunta norteadora:** como identificar itens que aparecem juntos sem confundir frequência, capacidade de previsão e causalidade?


## Objetivos de aprendizagem

Ao concluir este notebook, você será capaz de:

- representar transações como conjuntos e matriz *one-hot*;
- definir itemset, suporte e regra de associação;
- calcular suporte, confiança e *lift* manualmente;
- gerar e filtrar regras sem atribuir causalidade indevida.


In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder


## Dados transacionais

Cada transação é um conjunto de itens, sem quantidade nem ordem. A matriz *one-hot* possui uma linha por compra e uma coluna booleana por item. Essa representação responde apenas se o item ocorreu; sequência e quantidade exigem outras estruturas.


In [2]:
transacoes = [
    ["arroz", "feijao", "oleo"], ["arroz", "feijao"],
    ["arroz", "leite"], ["pao", "leite", "manteiga"],
    ["pao", "leite"], ["arroz", "feijao", "oleo"],
    ["pao", "cafe"], ["arroz", "feijao", "leite"],
    ["pao", "leite", "manteiga"], ["arroz", "feijao"],
    ["cafe", "leite"], ["arroz", "feijao", "oleo", "leite"],
]
codificador = TransactionEncoder()
matriz = codificador.fit(transacoes).transform(transacoes)
cestas = pd.DataFrame(matriz, columns=codificador.columns_)
cestas.index = pd.Index(range(1, len(cestas) + 1), name="transacao")
cestas


,arroz,cafe,feijao,leite,manteiga,oleo,pao
transacao,,,,,,,
1,True,False,True,False,False,True,False
2,True,False,True,False,False,False,False
3,True,False,False,True,False,False,False
4,False,False,False,True,True,False,True
5,False,False,False,True,False,False,True
6,True,False,True,False,False,True,False
7,False,True,False,False,False,False,True
8,True,False,True,True,False,False,False
9,False,False,False,True,True,False,True


## Suporte, confiança e *lift*: três perguntas diferentes

Em uma regra de associação

$$A\rightarrow B,$$

o antecedente $A$ e o consequente $B$ são itemsets disjuntos. A leitura é:

> **Se $A$ ocorre em uma transação, então $B$ tende a ocorrer na mesma transação.**

Suporte, confiança e *lift* avaliam aspectos diferentes da regra. Para compará-los, usaremos inicialmente um exemplo com 100 compras: café aparece em 40, açúcar em 50 e ambos aparecem juntos em 30.

### 1. Suporte

O **suporte de um itemset** é a proporção de transações que contém todos os seus itens. Para $N$ transações:

$$\operatorname{sup}(A)=\frac{\#(A)}{N}.$$

O **suporte da regra** $A\rightarrow B$ é o suporte da união $A\cup B$:

$$\operatorname{sup}(A\rightarrow B)=\operatorname{sup}(A\cup B)=P(A\cap B).$$

Ele responde à pergunta: **em que proporção de todas as transações $A$ e $B$ aparecem juntos?**

#### Interpretação

- suporte próximo de **0**: a combinação é rara na base;
- suporte maior: a combinação envolve uma parcela maior das transações;
- não existe um valor universalmente alto ou baixo: o limiar depende do tamanho da base, do domínio e do custo de analisar padrões raros.

#### Exemplo rápido

Para $\{\text{café}\}\rightarrow\{\text{açúcar}\}$, os dois produtos aparecem juntos em 30 das 100 compras:

$$\operatorname{sup}(\text{café}\rightarrow\text{açúcar})=\frac{30}{100}=0{,}30.$$

Portanto, a regra abrange **30% de todas as compras**. O suporte não diz, sozinho, se açúcar é especialmente frequente quando café ocorre; ele mede apenas o alcance conjunto da regra.

### 2. Confiança

A **confiança** mede a proporção das transações com $A$ que também contêm $B$:

$$\operatorname{Confiança}(A\rightarrow B)=\frac{\operatorname{sup}(A\cup B)}{\operatorname{sup}(A)}=P(B\mid A).$$

Ela responde: **entre as transações em que $A$ ocorre, em qual proporção $B$ também ocorre?**

#### Interpretação

- **Confiança = 0:** nenhuma transação com $A$ contém $B$;
- **0 < Confiança < 1:** apenas parte das ocorrências de $A$ é acompanhada por $B$;
- **Confiança = 1:** todas as ocorrências de $A$ na amostra são acompanhadas por $B$.

A confiança é **direcional**. Em geral, $\operatorname{Confiança}(A\rightarrow B)$ é diferente de $\operatorname{Confiança}(B\rightarrow A)$, pois os denominadores são diferentes.

#### Exemplo rápido

Das 40 compras com café, 30 também contêm açúcar:

$$\operatorname{Confiança}(\text{café}\rightarrow\text{açúcar})=\frac{30}{40}=0{,}75.$$

Assim, **75% das compras com café também incluem açúcar**. Contudo, uma confiança elevada pode ocorrer simplesmente porque o consequente é muito comum. Para avaliar esse efeito, precisamos comparar 75% com a frequência geral do açúcar.

### 3. *Lift*

O ***lift*** compara a confiança da regra com o suporte do consequente:

$$\operatorname{Lift}(A\rightarrow B)=\frac{\operatorname{Confiança}(A\rightarrow B)}{\operatorname{sup}(B)}=\frac{P(B\mid A)}{P(B)}.$$

Ele responde: **quantas vezes $B$ é mais ou menos frequente quando $A$ ocorre, em comparação com a frequência geral de $B$?**

#### Interpretação

- **Lift = 1:** $A$ não altera a frequência relativa de $B$; o resultado coincide com a independência;
- **Lift > 1:** associação positiva; $B$ é relativamente mais frequente quando $A$ ocorre;
- **Lift < 1:** associação negativa; $B$ é relativamente menos frequente quando $A$ ocorre.

#### Exemplo rápido

Açúcar aparece em 50% de todas as compras, mas em 75% das compras com café:

$$\operatorname{Lift}(\text{café}\rightarrow\text{açúcar})=\frac{0{,}75}{0{,}50}=1{,}5.$$

A presença de açúcar é, portanto, **1,5 vez tão frequente** entre as compras com café quanto na base completa — um aumento relativo de 50%. Isso não significa aumento de 50 pontos percentuais: a frequência passou da taxa-base de 50% para 75%, diferença de 25 pontos percentuais.

O *lift* é simétrico, embora sua fórmula pareça direcional:

$$\operatorname{Lift}(A\rightarrow B)=\frac{P(A\cap B)}{P(A)P(B)}=\operatorname{Lift}(B\rightarrow A).$$

### Resumo comparativo

| Métrica | O que mede | Referência principal | Direção da regra |
|---|---|---|---|
| **Suporte** | frequência conjunta de $A$ e $B$ em toda a base | 0 significa nenhuma coocorrência | simétrico |
| **Confiança** | frequência de $B$ dentro das transações com $A$ | varia de 0 a 1 | direcional |
| ***Lift*** | frequência relativa de $B$ com $A$ comparada à taxa-base de $B$ | 1 representa independência | simétrico |

Em termos intuitivos:

> **Suporte:** com que frequência $A$ e $B$ aparecem juntos em toda a base?

> **Confiança:** quando $A$ aparece, com que frequência $B$ também aparece?

> ***Lift*:** essa frequência é maior ou menor do que a frequência geral de $B$?

As três métricas descrevem associações observadas. Nenhuma delas demonstra que $A$ causa $B$.


In [3]:
def suporte(itens: set[str], compras: list[list[str]]) -> float:
    """Calcula a fração de transações que contêm todos os itens."""
    return sum(itens.issubset(set(compra)) for compra in compras) / len(compras)


x, y = {"feijao"}, {"arroz"}
suporte_x = suporte(x, transacoes)
suporte_y = suporte(y, transacoes)
suporte_xy = suporte(x | y, transacoes)
confianca = suporte_xy / suporte_x
lift = confianca / suporte_y
confianca_inversa = suporte_xy / suporte_y
lift_inverso = confianca_inversa / suporte_x
display(pd.Series({
    "numero_de_transacoes": len(transacoes),
    "transacoes_com_feijao": int(suporte_x * len(transacoes)),
    "transacoes_com_arroz": int(suporte_y * len(transacoes)),
    "transacoes_com_ambos": int(suporte_xy * len(transacoes)),
}, name="contagem"))
pd.Series({
    "suporte(feijao)": suporte_x, "suporte(arroz)": suporte_y,
    "suporte(feijao, arroz)": suporte_xy,
    "confianca(feijao -> arroz)": confianca,
    "lift(feijao -> arroz)": lift,
    "confianca(arroz -> feijao)": confianca_inversa,
    "lift(arroz -> feijao)": lift_inverso,
}, name="valor").round(3)


numero_de_transacoes     12
transacoes_com_feijao     6
transacoes_com_arroz      7
transacoes_com_ambos      6
Name: contagem, dtype: int64

suporte(feijao)               0.500
suporte(arroz)                0.583
suporte(feijao, arroz)        0.500
confianca(feijao -> arroz)    1.000
lift(feijao -> arroz)         1.714
confianca(arroz -> feijao)    0.857
lift(arroz -> feijao)         1.714
Name: valor, dtype: float64

### Aplicação à regra $\{feijao\}\rightarrow\{arroz\}$

#### Suporte

Feijão e arroz aparecem juntos em 6 das 12 transações:

$$\operatorname{sup}(\text{feijão}\rightarrow\text{arroz})=\frac{6}{12}=0{,}50.$$

Isso significa que a regra abrange **50% de todas as compras**. O mesmo suporte vale para a regra inversa, pois ambas representam a mesma coocorrência $\{\text{arroz},\text{feijão}\}$.

#### Confiança

Feijão aparece em 6 transações, e todas as 6 também contêm arroz:

$$\operatorname{Confiança}(\text{feijão}\rightarrow\text{arroz})=\frac{6}{6}=1.$$

Portanto, **100% das compras com feijão contêm arroz nesta amostra**. Na direção inversa, arroz aparece em 7 transações e 6 contêm feijão:

$$\operatorname{Confiança}(\text{arroz}\rightarrow\text{feijão})=\frac{6}{7}\approx0{,}857.$$

A diferença confirma que a confiança é direcional: existe uma compra com arroz sem feijão, mas nenhuma compra com feijão sem arroz.

#### *Lift*

Arroz aparece em $7/12\approx0{,}583$ de todas as compras. Comparando a confiança 1 com essa taxa-base:

$$\operatorname{Lift}(\text{feijão}\rightarrow\text{arroz})=\frac{1}{7/12}=\frac{12}{7}\approx1{,}714.$$

Assim, arroz é aproximadamente **1,714 vez tão frequente nas compras com feijão** quanto na base completa, um aumento relativo de cerca de 71,4%. A regra inversa possui o mesmo *lift*, porque $6/12\div[(7/12)(6/12)]$ também resulta em $12/7$.

Confiança 1 e *lift* acima de 1 descrevem esta amostra pequena; não garantem que toda compra futura com feijão terá arroz e não provam que feijão causa a compra de arroz. Promoções, hábitos de consumo e organização da loja podem explicar a associação.


## Mineração e filtragem

Primeiro encontramos itemsets que atendem ao suporte mínimo; depois geramos regras. Limiares são decisões do problema: suporte alto pode ocultar nichos, enquanto suporte muito baixo pode produzir coincidências instáveis.


In [4]:
itemsets_frequentes = apriori(cestas, min_support=0.15, use_colnames=True)
regras = association_rules(
    itemsets_frequentes, metric="confidence", min_threshold=0.60
)
regras_selecionadas = (
    regras.loc[:, ["antecedents", "consequents", "support", "confidence", "lift"]]
    .sort_values(["lift", "support"], ascending=False)
)
regras_selecionadas.head(10).round(3)


,antecedents,consequents,support,confidence,lift
14,"(pao, leite)",(manteiga),0.167,0.667,4.000
15,(manteiga),"(pao, leite)",0.167,1.000,4.000
6,(manteiga),(pao),0.167,1.000,3.000
13,"(manteiga, leite)",(pao),0.167,1.000,3.000
3,(oleo),(feijao),0.250,1.000,2.000
10,"(arroz, oleo)",(feijao),0.250,1.000,2.000
11,(oleo),"(feijao, arroz)",0.250,1.000,2.000
0,(feijao),(arroz),0.500,1.000,1.714
1,(arroz),(feijao),0.500,0.857,1.714
2,(oleo),(arroz),0.250,1.000,1.714


Regras invertidas têm o mesmo suporte e *lift*, mas podem ter confianças diferentes. Antes de agir, verifique volume, estabilidade, redundância, custos, disponibilidade e explicações alternativas, como promoções e sazonalidade.

> **U04-NB01-V01 — Verifique seu entendimento:** por que uma regra com confiança de 90% pode ter *lift* menor que 1?

> **U04-NB01-E01 — Exercício:** calcule manualmente suporte, confiança e *lift* para `pao -> leite` e para a regra inversa. Compare as confianças e explique por que o *lift* é igual nas duas direções.


## Síntese

- Dados transacionais representam presença de itens por evento.
- Suporte mede prevalência; confiança é condicional; *lift* compara com a taxa-base.
- Regras fortes podem ser raras, redundantes, instáveis ou sem utilidade.
- Associação observada não demonstra causalidade.

## Referências

- HAN, Jiawei; PEI, Jian; TONG, Hanghang. *Data Mining: Concepts and Techniques*. 4. ed. Cambridge: Morgan Kaufmann/Elsevier, 2023. Cap. 4, seções 4.1–4.2.
- RASCHKA, Sebastian. *mlxtend documentation*: frequent patterns. Versão 0.23.
